# AutoResearch: Cheatsheet Optimization Analysis

Analyze results from the autonomous cheatsheet optimization loop.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

df = pd.read_csv("results.tsv", sep="\t")
df["accuracy"] = pd.to_numeric(df["accuracy"], errors="coerce")
df["cheatsheet_kb"] = pd.to_numeric(df["cheatsheet_kb"], errors="coerce")
df["status"] = df["status"].str.strip().str.upper()

print(f"Total experiments: {len(df)}")
print(f"Columns: {list(df.columns)}")
df.head(10)

In [ ]:
counts = df["status"].value_counts()
print("Experiment outcomes:")
print(counts.to_string())

n_keep = counts.get("KEEP", 0)
n_discard = counts.get("DISCARD", 0)
n_error = counts.get("ERROR", 0)
n_decided = n_keep + n_discard
if n_decided > 0:
    print(f"\nKeep rate: {n_keep}/{n_decided} = {n_keep / n_decided:.1%}")

In [ ]:
# Show all KEPT experiments (improvements that stuck)
kept = df[df["status"] == "KEEP"].copy()
print(f"KEPT experiments ({len(kept)} total):\n")
for i, row in kept.iterrows():
    acc = row["accuracy"]
    desc = row["description"]
    print(f"  #{i:3d}  acc={acc:.3f}  size={row['cheatsheet_kb']:.1f}KB  {desc}")

## Accuracy Over Time

Track how accuracy evolves. Running maximum shows the frontier.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))

# Filter out errors
valid = df[df["status"] != "ERROR"].copy().reset_index(drop=True)
baseline_acc = valid.loc[0, "accuracy"]

# Discarded as faint background
disc = valid[valid["status"] == "DISCARD"]
ax.scatter(disc.index, disc["accuracy"],
           c="#cccccc", s=12, alpha=0.5, zorder=2, label="Discarded")

# Kept as prominent green
kept_v = valid[valid["status"] == "KEEP"]
ax.scatter(kept_v.index, kept_v["accuracy"],
           c="#2ecc71", s=50, zorder=4, label="Kept", edgecolors="black", linewidths=0.5)

# Running maximum
kept_mask = valid["status"] == "KEEP"
kept_idx = valid.index[kept_mask]
kept_acc = valid.loc[kept_mask, "accuracy"]
running_max = kept_acc.cummax()
ax.step(kept_idx, running_max, where="post", color="#27ae60",
        linewidth=2, alpha=0.7, zorder=3, label="Running best")

# Label each kept experiment
for idx, acc in zip(kept_idx, kept_acc):
    desc = str(valid.loc[idx, "description"]).strip()
    if len(desc) > 45:
        desc = desc[:42] + "..."
    ax.annotate(desc, (idx, acc),
                textcoords="offset points",
                xytext=(6, 6), fontsize=8.0,
                color="#1a7a3a", alpha=0.9,
                rotation=30, ha="left", va="bottom")

n_total = len(df)
n_kept = len(df[df["status"] == "KEEP"])
best = kept_acc.max() if len(kept_acc) > 0 else baseline_acc

ax.set_xlabel("Experiment #", fontsize=12)
ax.set_ylabel("Accuracy (higher is better)", fontsize=12)
ax.set_title(f"AutoResearch Progress: {n_total} Experiments, {n_kept} Kept Improvements", fontsize=14)
ax.legend(loc="lower right", fontsize=9)
ax.grid(True, alpha=0.2)

margin = (best - baseline_acc) * 0.15 if best > baseline_acc else 0.02
ax.set_ylim(baseline_acc - margin, best + margin)

plt.tight_layout()
plt.savefig("progress.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to progress.png")

## Summary Statistics

In [ ]:
kept = df[df["status"] == "KEEP"].copy()
baseline_acc = df.iloc[0]["accuracy"]
best_acc = kept["accuracy"].max()
best_row = kept.loc[kept["accuracy"].idxmax()]

print(f"Baseline accuracy:  {baseline_acc:.3f}")
print(f"Best accuracy:      {best_acc:.3f}")
print(f"Total improvement:  {best_acc - baseline_acc:+.3f} ({(best_acc - baseline_acc) / baseline_acc * 100:+.1f}%)")
print(f"Best experiment:    {best_row['description']}")
print(f"Best cheatsheet:    {best_row['cheatsheet_kb']:.1f} KB")
print()

print("Cumulative improvements:")
for i, (_, row) in enumerate(kept.iterrows()):
    print(f"  #{i}: acc={row['accuracy']:.3f}  {row['cheatsheet_kb']:.1f}KB  {row['description']}")

## Top Hits (by improvement delta)

In [ ]:
kept = df[df["status"] == "KEEP"].copy()
kept["prev_acc"] = kept["accuracy"].shift(1)
kept["delta"] = kept["accuracy"] - kept["prev_acc"]

hits = kept.iloc[1:].copy()
hits = hits.sort_values("delta", ascending=False)

print(f"{'Rank':>4}  {'Delta':>8}  {'Acc':>8}  Description")
print("-" * 80)
for rank, (_, row) in enumerate(hits.iterrows(), 1):
    print(f"{rank:4d}  {row['delta']:+.3f}    {row['accuracy']:.3f}    {row['description']}")

print(f"\n{'':>4}  {hits['delta'].sum():+.3f}    {'':>8}  TOTAL improvement over baseline")